# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [1]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.ensemble import VotingClassifier, BaggingClassifier, StackingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np
import gdown

In [2]:
url='https://drive.google.com/uc?id=1cHSxB__ITTlCvsEzbIwuxy3La4G6S3PB&export=download'

output='day-week_dataset.csv'
gdown.download(url, output, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1cHSxB__ITTlCvsEzbIwuxy3La4G6S3PB&export=download
To: /home/mirshod/Desktop/DSB11_ML_Advanced.ID_886524-1/src/ex03/day-week_dataset.csv
100%|██████████| 290k/290k [00:01<00:00, 237kB/s]


'day-week_dataset.csv'

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [3]:
df=pd.read_csv('day-week_dataset.csv')
df.head()

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4


In [4]:
X=df.drop(columns=['dayofweek'])
y=df['dayofweek']<5

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [6]:
def show_scores(model):
  acc_score=accuracy_score(y_test, model.predict(X_test))
  precision_val=precision_score(y_test, model.predict(X_test), average='weighted')
  recall_val=recall_score(y_test, model.predict(X_test), average='weighted')

  print("===\tSCORES\t===")
  print('accuracy is ', acc_score)
  print('precision is ', precision_val)
  print('recall is ', recall_val)

In [7]:
svm=SVC(kernel='rbf', C=10, class_weight='balanced', gamma='auto', probability=True,
    random_state=21)
svm.fit(X_train, y_train)

SVC(C=10, class_weight='balanced', gamma='auto', probability=True,
    random_state=21)

In [8]:
show_scores(svm)

===	SCORES	===
accuracy is  0.9408284023668639
precision is  0.9426051088102819
recall is  0.9408284023668639


In [9]:
decision_tree=DecisionTreeClassifier(criterion='gini', max_depth=22, random_state=21)
decision_tree.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=22, random_state=21)

In [10]:
show_scores(decision_tree)

===	SCORES	===
accuracy is  0.9497041420118343
precision is  0.9498016315082519
recall is  0.9497041420118343


In [11]:
random_forest=RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=36, n_estimators=10)
random_forest.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=36, n_estimators=10)

In [12]:
show_scores(random_forest)

===	SCORES	===
accuracy is  0.9467455621301775
precision is  0.9473486572598998
recall is  0.9467455621301775


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [13]:
voting=VotingClassifier(estimators=[('svm', svm), ('decision_tree', decision_tree), ('random_forest', random_forest)], voting = 'soft')
voting.fit(X_train, y_train)

VotingClassifier(estimators=[('svm',
                              SVC(C=10, class_weight='balanced', gamma='auto',
                                  probability=True, random_state=21)),
                             ('decision_tree',
                              DecisionTreeClassifier(max_depth=22,
                                                     random_state=21)),
                             ('random_forest',
                              RandomForestClassifier(class_weight='balanced',
                                                     criterion='entropy',
                                                     max_depth=36,
                                                     n_estimators=10))],
                 voting='soft')

In [14]:
show_scores(voting)

===	SCORES	===
accuracy is  0.9615384615384616
precision is  0.9618765567357979
recall is  0.9615384615384616


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [15]:
bagging=BaggingClassifier(estimator=svm, n_estimators=100, random_state=21)
bagging.fit(X_train, y_train)

BaggingClassifier(estimator=SVC(C=10, class_weight='balanced', gamma='auto',
                                probability=True, random_state=21),
                  n_estimators=100, random_state=21)

In [16]:
show_scores(bagging)

===	SCORES	===
accuracy is  0.9378698224852071
precision is  0.9379858603795841
recall is  0.9378698224852071


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [17]:
estimators=[('svm', svm), ('decision_tree', decision_tree), ('random_forest', random_forest)]
final_est=LogisticRegression(solver='liblinear')
n_splits=[2, 3, 4, 5, 6, 7]
passthrough=[True, False]

In [18]:
for n in n_splits:
  for p in passthrough:
    skf=StratifiedKFold(n_splits=n, shuffle=True, random_state=21)
    stacked_classifier=StackingClassifier(estimators=estimators, final_estimator=final_est, cv=skf, passthrough=p)
    stacked_classifier.fit(X_train, y_train)
    print("===\tRESULTS\t===")
    print('n_splits: ', n)
    print('passthrough: ', p)
    show_scores(stacked_classifier)


===	RESULTS	===
n_splits:  2
passthrough:  True
===	SCORES	===
accuracy is  0.9763313609467456
precision is  0.9767637687756032
recall is  0.9763313609467456
===	RESULTS	===
n_splits:  2
passthrough:  False
===	SCORES	===
accuracy is  0.9674556213017751
precision is  0.9675252882012536
recall is  0.9674556213017751
===	RESULTS	===
n_splits:  3
passthrough:  True
===	SCORES	===
accuracy is  0.9585798816568047
precision is  0.9591147018661811
recall is  0.9585798816568047
===	RESULTS	===
n_splits:  3
passthrough:  False
===	SCORES	===
accuracy is  0.9556213017751479
precision is  0.9557095170725858
recall is  0.9556213017751479
===	RESULTS	===
n_splits:  4
passthrough:  True
===	SCORES	===
accuracy is  0.9733727810650887
precision is  0.9740203147744826
recall is  0.9733727810650887
===	RESULTS	===
n_splits:  4
passthrough:  False
===	SCORES	===
accuracy is  0.9556213017751479
precision is  0.9559857335019749
recall is  0.9556213017751479
===	RESULTS	===
n_splits:  5
passthrough:  True
=

## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [19]:
cv=StratifiedKFold(n_splits=7, shuffle=True, random_state=21)
stacked_classifer=StackingClassifier(estimators=estimators, final_estimator=final_est, cv=cv, passthrough=False)
stacked_classifier.fit(X_train, y_train)

StackingClassifier(cv=StratifiedKFold(n_splits=7, random_state=21, shuffle=True),
                   estimators=[('svm',
                                SVC(C=10, class_weight='balanced', gamma='auto',
                                    probability=True, random_state=21)),
                               ('decision_tree',
                                DecisionTreeClassifier(max_depth=22,
                                                       random_state=21)),
                               ('random_forest',
                                RandomForestClassifier(class_weight='balanced',
                                                       criterion='entropy',
                                                       max_depth=36,
                                                       n_estimators=10))],
                   final_estimator=LogisticRegression(solver='liblinear'))

In [20]:
pred=stacked_classifier.predict(X)
pred=pd.Series(pred)
pred

0       True
1       True
2       True
3       True
4       True
        ... 
1681    True
1682    True
1683    True
1684    True
1685    True
Length: 1686, dtype: bool

In [21]:
df['error']=(y!=pred)

day_name_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
df['day_name'] = df['dayofweek'].apply(lambda x: day_name_map[x])

errors_per_day = df.groupby('day_name')['error'].sum()
total_samples_per_day = df.groupby('day_name').size()

error_percentage_per_day = (errors_per_day / total_samples_per_day) * 100

analysis_df = pd.DataFrame({
    'total_samples': total_samples_per_day,
    'total_errors': errors_per_day,
    'error_percentage': error_percentage_per_day
})

analysis_df.sort_values(by='error_percentage', ascending=False, inplace=True)
display(analysis_df)


,total_samples,total_errors,error_percentage
day_name,,,
Monday,136,2,1.470588
Thursday,396,5,1.262626
Sunday,356,4,1.123596
Saturday,271,3,1.107011
Friday,104,1,0.961538
Tuesday,274,1,0.364964
Wednesday,149,0,0.000000


In [22]:
import joblib
joblib.dump(stacked_classifer, 'stacked_classifer.joblib')

['stacked_classifer.joblib']